# Brasil API - Banks
Este notebook contém os testes automatizados para o tema **Banks** da Brasil API.

### Estrutura de Testes
O objetivo é validar o funcionamento dos endpoints, garantindo que os dados retornados estejam corretos e que erros sejam tratados adequadamente.

## 🛠️ Configuração Inicial
Importação de bibliotecas e definição da função `run_test` para padronização dos resultados.

In [ ]:
import requests
import json
from datetime import datetime

BASE = "https://brasilapi.com.br/api"

def run_test(name, condition, success_message, failure_message, response_data=None):
    if condition:
        print(f"✅ {name}: {success_message}")
        if response_data is not None:
            # Limita a exibição do JSON para não poluir o notebook
            json_output = json.dumps(response_data, indent=2, ensure_ascii=False)
            if len(json_output) > 500:
                print(f"   ↳ Dados da resposta (primeiros 500 caracteres):\n{json_output[:500]}...")
            else:
                print(f"   ↳ Dados da resposta:\n{json_output}")
        return True
    else:
        print(f"❌ {name}: {failure_message}")
        return False


## 🧪 Execução dos Testes: Banks

*   Item da lista
*   Item da lista


Validação do endpoint de bancos para garantir o retorno de instituições e tratamento de erros.

In [ ]:

# Teste 1: Listar todos os bancos
try:
    response_banks = requests.get(f"{BASE}/banks/v1", timeout=10)
    test1_condition = response_banks.status_code == 200 and isinstance(response_banks.json(), list) and len(response_banks.json()) > 0
    run_test(
        "Listar todos os bancos",
        test1_condition,
        "Status 200 e lista não vazia recebida.",
        f"Falha: Status {response_banks.status_code} ou lista vazia/formato incorreto.",
        response_banks.json() if response_banks.status_code == 200 else None
    )
except Exception as e:
    run_test("Listar todos os bancos", False, "", f"Erro: {e}")

# Teste 2: Buscar banco pelo código (código 1)
try:
    response_bank_id = requests.get(f"{BASE}/banks/v1/1", timeout=10)
    test2_condition = response_bank_id.status_code == 200 and response_bank_id.json().get("code") == 1
    run_test(
        "Buscar banco pelo código (ID 1)",
        test2_condition,
        "Status 200 e banco com ID 1 encontrado.",
        f"Falha: Status {response_bank_id.status_code} ou ID do banco incorreto.",
        response_bank_id.json() if response_bank_id.status_code == 200 else None
    )
except Exception as e:
    run_test("Buscar banco pelo código (ID 1)", False, "", f"Erro: {e}")

# Teste 3: Banco inexistente
try:
    response_none = requests.get(f"{BASE}/banks/v1/99999", timeout=10)
    run_test(
        "Banco inexistente",
        response_none.status_code == 404,
        "Status 404 retornado corretamente.",
        f"Falha: Status {response_none.status_code} em vez de 404.",
        response_none.json() if response_none.status_code == 200 else None
    )
except Exception as e:
    run_test("Banco inexistente", False, "", f"Erro: {e}")

print("\n--- Testes Concluídos ---")

✅ Listar todos os bancos: Status 200 e lista não vazia recebida.
   ↳ Dados da resposta (primeiros 500 caracteres):
[
  {
    "ispb": "00000000",
    "name": "BCO DO BRASIL S.A.",
    "code": 1,
    "fullName": "Banco do Brasil S.A."
  },
  {
    "ispb": "00000208",
    "name": "BRB - BCO DE BRASILIA S.A.",
    "code": 70,
    "fullName": "BRB - BANCO DE BRASILIA S.A."
  },
  {
    "ispb": "00038121",
    "name": "Selic",
    "code": null,
    "fullName": "Banco Central do Brasil - Selic"
  },
  {
    "ispb": "00038166",
    "name": "Bacen",
    "code": null,
    "fullName": "Banco Central do Brasil"
  },
  {...
✅ Buscar banco pelo código (ID 1): Status 200 e banco com ID 1 encontrado.
   ↳ Dados da resposta:
{
  "ispb": "00000000",
  "name": "BCO DO BRASIL S.A.",
  "code": 1,
  "fullName": "Banco do Brasil S.A."
}
✅ Banco inexistente: Status 404 retornado corretamente.

--- Testes Concluídos ---


## 🧪 Execução dos Testes: CNPJ
Validação de CNPJ para garantir que dados reais sejam retornados e entradas inválidas gerem erro.

In [ ]:

# Teste 1: CNPJ válido
try:
    response = requests.get(f"{BASE}/cnpj/v1/19131243000197", timeout=10)
    data = response.json() if response.status_code == 200 else {}
    run_test(
        "CNPJ válido",
        response.status_code == 200,
        "Status 200 e dados do CNPJ recebidos.",
        f"Falha: Status {response.status_code}.",
        data
    )

    if response.status_code == 200:
        campos = all(c in data for c in ["cnpj", "razao_social", "descricao_situacao_cadastral"])
        run_test(
            "Campos obrigatórios",
            campos,
            "Todos os campos obrigatórios presentes.",
            f"Campos ausentes. Encontrados: {list(data.keys())[:5]}"
        )
except Exception as e:
    run_test("CNPJ válido", False, "", f"Erro: {e}")

# Teste 2: CNPJ inválido
try:
    response_inv = requests.get(f"{BASE}/cnpj/v1/00000000000000", timeout=10)
    run_test(
        "CNPJ inválido",
        response_inv.status_code in [400, 404, 422, 500],
        f"Erro tratado corretamente (Status {response_inv.status_code}).",
        f"Falha: Status {response_inv.status_code} não esperado para entrada inválida."
    )
except Exception as e:
    run_test("CNPJ inválido", False, "", f"Erro: {e}")

print("\n--- Testes Concluídos ---")

✅ CNPJ válido: Status 200 e dados do CNPJ recebidos.
   ↳ Dados da resposta (primeiros 500 caracteres):
{
  "uf": "SP",
  "cep": "01311902",
  "qsa": [
    {
      "pais": null,
      "nome_socio": "HAYDEE SVAB",
      "codigo_pais": null,
      "faixa_etaria": "Entre 41 a 50 anos",
      "cnpj_cpf_do_socio": "***112108**",
      "qualificacao_socio": "Presidente",
      "codigo_faixa_etaria": 5,
      "data_entrada_sociedade": "2024-02-27",
      "identificador_de_socio": 2,
      "cpf_representante_legal": "***000000**",
      "nome_representante_legal": "",
      "codigo_qualificacao_socio": 16...
✅ Campos obrigatórios: Todos os campos obrigatórios presentes.
✅ CNPJ inválido: Erro tratado corretamente (Status 404).

--- Testes Concluídos ---


## 🧪 Execução dos Testes: CEP
Busca de CEP para validar retorno de endereço e geolocalização.

In [ ]:

# Teste 1: CEP válido (v1)
try:
    response = requests.get(f"{BASE}/cep/v1/01310100", timeout=10)
    data = response.json() if response.status_code == 200 else {}
    run_test(
        "CEP válido (v1)",
        response.status_code == 200,
        "Status 200 e endereço retornado.",
        f"Falha: Status {response.status_code}.",
        data
    )
    if response.status_code == 200:
        run_test("UF válida", len(data.get("state", "")) == 2, "Estado com 2 caracteres.", "Estado inválido.")
except Exception as e:
    run_test("CEP válido (v1)", False, "", f"Erro: {e}")

# Teste 2: CEP com geolocalização (v2)
try:
    response_v2 = requests.get(f"{BASE}/cep/v2/01310100", timeout=10)
    data_v2 = response_v2.json() if response_v2.status_code == 200 else {}
    run_test(
        "CEP com geolocalização (v2)",
        response_v2.status_code == 200 and "location" in data_v2,
        "Status 200 e dados de localização presentes.",
        f"Falha: Status {response_v2.status_code} ou campo 'location' ausente.",
        data_v2
    )
except Exception as e:
    run_test("CEP com geolocalização (v2)", False, "", f"Erro: {e}")

print("\n--- Testes Concluídos ---")

✅ CEP válido (v1): Status 200 e endereço retornado.
   ↳ Dados da resposta:
{
  "cep": "01310100",
  "state": "SP",
  "city": "São Paulo",
  "neighborhood": "Bela Vista",
  "street": "Avenida Paulista",
  "service": "open-cep"
}
✅ UF válida: Estado com 2 caracteres.
✅ CEP com geolocalização (v2): Status 200 e dados de localização presentes.
   ↳ Dados da resposta:
{
  "cep": "01310100",
  "state": "SP",
  "city": "São Paulo",
  "neighborhood": "Bela Vista",
  "street": "Avenida Paulista",
  "service": "open-cep",
  "timezoneName": null,
  "location": {
    "type": "Point",
    "coordinates": {}
  }
}

--- Testes Concluídos ---


## 🧪 Execução dos Testes: DDD
Validação do endpoint de DDD para garantir que o estado e cidades correspondam ao informado.

In [ ]:

# Teste 1: DDD 11 (SP)
try:
    response = requests.get(f"{BASE}/ddd/v1/11", timeout=10)
    data = response.json() if response.status_code == 200 else {}
    run_test(
        "DDD 11",
        response.status_code == 200 and data.get("state") == "SP",
        "Status 200 e estado SP identificado.",
        f"Falha: Status {response.status_code} ou estado incorreto.",
        data
    )
except Exception as e:
    run_test("DDD 11", False, "", f"Erro: {e}")

# Teste 2: DDD 21 (RJ)
try:
    response = requests.get(f"{BASE}/ddd/v1/21", timeout=10)
    data = response.json() if response.status_code == 200 else {}
    run_test(
        "DDD 21",
        response.status_code == 200 and data.get("state") == "RJ",
        "Status 200 e estado RJ identificado.",
        f"Falha: Status {response.status_code} ou estado incorreto.",
        data
    )
except Exception as e:
    run_test("DDD 21", False, "", f"Erro: {e}")

print("\n--- Testes Concluídos ---")

✅ DDD 11: Status 200 e estado SP identificado.
   ↳ Dados da resposta (primeiros 500 caracteres):
{
  "state": "SP",
  "cities": [
    "EMBU",
    "VÁRZEA PAULISTA",
    "VARGEM GRANDE PAULISTA",
    "VARGEM",
    "TUIUTI",
    "TABOÃO DA SERRA",
    "SUZANO",
    "SÃO ROQUE",
    "SÃO PAULO",
    "SÃO LOURENÇO DA SERRA",
    "SÃO CAETANO DO SUL",
    "SÃO BERNARDO DO CAMPO",
    "SANTO ANDRÉ",
    "SANTANA DE PARNAÍBA",
    "SANTA ISABEL",
    "SALTO",
    "SALESÓPOLIS",
    "RIO GRANDE DA SERRA",
    "RIBEIRÃO PIRES",
    "POÁ",
    "PIRAPORA DO BOM JESUS",
    "PIRACAIA",
    "PINHALZINHO...
✅ DDD 21: Status 200 e estado RJ identificado.
   ↳ Dados da resposta:
{
  "state": "RJ",
  "cities": [
    "TERESÓPOLIS",
    "TANGUÁ",
    "SEROPÉDICA",
    "SÃO JOÃO DE MERITI",
    "SÃO GONÇALO",
    "RIO DE JANEIRO",
    "RIO BONITO",
    "QUEIMADOS",
    "PARACAMBI",
    "NOVA IGUAÇU",
    "NITERÓI",
    "NILÓPOLIS",
    "MESQUITA",
    "MARICÁ",
    "MANGARATIBA",
    "MAGÉ",
    "JAPERI"

## 🧪 Execução dos Testes: Feriados
Listagem de feriados nacionais para validar retorno de datas importantes.

In [ ]:

# Teste 1: Feriados do ano atual
try:
    ano = datetime.now().year
    response = requests.get(f"{BASE}/feriados/v1/{ano}", timeout=10)
    data = response.json() if response.status_code == 200 else []
    run_test(
        f"Feriados {ano}",
        response.status_code == 200 and isinstance(data, list),
        "Status 200 e lista de feriados recebida.",
        f"Falha: Status {response.status_code} ou formato inválido.",
        data
    )

    if response.status_code == 200 and isinstance(data, list):
        natal = next((f for f in data if f.get("date", "").endswith("-12-25")), None)
        run_test("Natal presente", natal is not None, "Feriado de Natal encontrado.", "Natal não listado.", natal)
except Exception as e:
    run_test("Feriados", False, "", f"Erro: {e}")

print("\n--- Testes Concluídos ---")

✅ Feriados 2026: Status 200 e lista de feriados recebida.
   ↳ Dados da resposta (primeiros 500 caracteres):
[
  {
    "date": "2026-01-01",
    "name": "Confraternização mundial",
    "type": "national",
    "weekday": "quinta-feira"
  },
  {
    "date": "2026-02-17",
    "name": "Carnaval",
    "type": "national",
    "weekday": "terça-feira"
  },
  {
    "date": "2026-04-03",
    "name": "Sexta-feira Santa",
    "type": "national",
    "weekday": "sexta-feira"
  },
  {
    "date": "2026-04-05",
    "name": "Páscoa",
    "type": "national",
    "weekday": "domingo"
  },
  {
    "date": "2026-04-21",
...
✅ Natal presente: Feriado de Natal encontrado.
   ↳ Dados da resposta:
{
  "date": "2026-12-25",
  "name": "Natal",
  "type": "national",
  "weekday": "sexta-feira"
}

--- Testes Concluídos ---
